# 02 — Voter registration 2026

**Goal:** one row per municipality: registered voters by age group and gender, with snapshot_date

**Reads:** `data/raw/registration/`  
**Writes:** `INTERIM / "registration_2026.csv"`

Run the two setup cells below first, every time.

#Setup

In [ ]:
import sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/Thato20-M/democratic-funnel.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content/democratic-funnel")
    url = REPO_URL
    try:                                    # private repo: token from Colab Secrets
        from google.colab import userdata
        url = REPO_URL.replace("https://", f"https://{userdata.get('GITHUB_TOKEN')}@")
    except Exception:
        print("No GITHUB_TOKEN in Colab Secrets - cloning will fail for the private repo")
    if REPO_DIR.exists():
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", url, str(REPO_DIR)], check=True)
else:
    REPO_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / "src" / "config.py").exists())

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print("Repo:", REPO_DIR)

Mounted at /content/drive
Repo: /content/democratic-funnel


In [ ]:
from src import config
from src.config import RAW, INTERIM, PROCESSED, MODELS, FIGURES, SEED
from src.io_utils import read_any, clean_columns, inventory
from src.privacy import drop_personal

import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import warnings; warnings.filterwarnings("ignore", category=FutureWarning)

config.make_dirs(); config.set_seed()
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid"); plt.rcParams["figure.dpi"] = 110
print(config.describe())

Environment: Colab
Repo root:   /content/democratic-funnel
Work root:   /content/drive/MyDrive/DIRISA_SDC


#How the data was acquired


## ENVIRONMENT SETUP & DEPENDENCY CONFIGURATION
 Objective: Prepare the cloud virtual machine (Google Colab) environment with the necessary web automation tools and data libraries required to extract real-time voter registration data from the official IEC portal.


In [ ]:
# Install required libraries and Chromium browser tools for Colab
!apt-get update!
!pip install webdriver-manager selenium pandas
!apt-get install -y chromium-browser chromium-chromedriver
!pip install -q playwright
!pip install chromium
!playwright install chromium

# Install Playwright Dependencies
!playwright install-deps





#**Testing Setup to access the IEC Data**


In [ ]:
import asyncio
from playwright.async_api import async_playwright

async def get_iec_page():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        await page.goto("https://www.elections.org.za/pw/StatsData/Voter-Registration-Statistics")
        print("Success! ", await page.title())

        await browser.close()

# Run the async function in Colab
await get_iec_page()

Success!  Voter Registration Statistics - Electoral Commission of South Africa


 HTTP CONNECTIVITY & API PROBING TEST

Objective: Test whether the target Electoral Commission of South Africa (IEC) statistical page is accessible via a lightweight HTTP GET request, and  determine if the page content is statically rendered (parseable via BeautifulSoup) or dynamically loaded via client-side JavaScript (requiring Selenium).

In [ ]:

import requests

# Test a direct request to see if the page responds or exposes an API
url = "https://www.elections.org.za/pw/StatsData/Voter-Registration-Statistics"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

response = requests.get(url, headers=headers)
print("Response error Code:", response.status_code)

if response.status_code == 200:
    print("Page is reachable")
else:
    print("check issues")

Response error Code: 200
Page is reachable



#**Acccessing Data in the filters (i.e by Province)**



In [ ]:
import asyncio
from playwright.async_api import async_playwright
import pandas as pd

async def extract_iec_data():
    async with async_playwright() as p:
        # Launch browser in headless mode
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        # Navigate to the dashboard
        print("Navigating to IEC Voter Registration portal...")
        await page.goto("https://www.elections.org.za/pw/StatsData/Voter-Registration-Statistics")

        # Wait for the main container or dropdowns to load
        await page.wait_for_selector("select", timeout=10000)
        print("Page loaded successfully.")

        # Find all select dropdowns on the page
        dropdowns = await page.locator("select").all()
        print(f"Found {len(dropdowns)} dropdown menus on the page.")

        # Example: Extracting province options from the first dropdown
        province_select = dropdowns[0]
        province_options = await province_select.locator("option").all_inner_texts()
        print("Provinces available:", province_options)

        # You can loop through provinces, select them, and extract the resulting data cards/tables
        extracted_data = []

        for province in province_options[1:]: # Skip "All provinces" option placeholder
            print(f"Selecting province: {province}")
            await province_select.select_option(label=province)

            # Wait a moment for dynamic content/municipality dropdown to update
            await page.wait_for_timeout(3000)

            # Grab summary text or demographic numbers displayed on the page
            # (Adjust selectors based on specific text elements you want to target)
            summary_cards = await page.locator(".col-md-6, .card, table").all_inner_texts()

            extracted_data.append({
                "Province": province,
                "Raw_Content": summary_cards[:3] # Capture the top summary text blocks
            })

        await browser.close()

        # Convert to DataFrame and save
        df = pd.DataFrame(extracted_data)
        df.to_csv("iec_extracted_snapshot.csv", index=False)
        print("Extraction complete! Saved to 'iec_extracted_snapshot.csv'.")

# Run the async scraper function in Colab
await extract_iec_data()

Navigating to IEC Voter Registration portal...
Page loaded successfully.
Found 2 dropdown menus on the page.
Provinces available: ['All provinces', 'Eastern Cape', 'Free State', 'Gauteng', 'KwaZulu-Natal', 'Limpopo', 'Mpumalanga', 'North West', 'Northern Cape', 'Western Cape']
Selecting province: Eastern Cape
Selecting province: Free State
Selecting province: Gauteng
Selecting province: KwaZulu-Natal
Selecting province: Limpopo
Selecting province: Mpumalanga
Selecting province: North West
Selecting province: Northern Cape
Selecting province: Western Cape
Extraction complete! Saved to 'iec_extracted_snapshot.csv'.


#***MUNICIPALITIES IN PROVINCES...  DATA EXTRACTION PIPELINE***

It opens the voter registration statistics page, then works through the province dropdown one at a time. For each province, it waits for the municipality dropdown to refresh, then loops through every municipality in that province.



In [ ]:
# @title
import asyncio
from playwright.async_api import async_playwright
import pandas as pd

async def extract_clean_iec_data():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        print("Opening IEC portal...")
        await page.goto("https://www.elections.org.za/pw/StatsData/Voter-Registration-Statistics")
        await page.wait_for_selector("select", timeout=15000)

        # Get province dropdown options
        province_options = await page.locator("select").first.locator("option").all_inner_texts()
        provinces = [p.strip() for p in province_options if p.strip() and p != "All provinces"]

        structured_data = []

        for province in provinces:
            print(f"Processing province: {province}")
            # Select province
            await page.locator("select").first.select_option(label=province)
            await page.wait_for_timeout(2000) # Wait for municipalities to update

            # Get municipality dropdown options for this province
            muni_dropdown = page.locator("select").nth(1)
            muni_options = await muni_dropdown.locator("option").all_inner_texts()
            municipalities = [m.strip() for m in muni_options if m.strip() and m != "All municipalities"]

            for muni in municipalities:
                print(f"  -> Selecting municipality: {muni}")
                await muni_dropdown.select_option(label=muni)
                await page.wait_for_timeout(2500) # Wait for demographic cards to load

                # Extract demographic statistics blocks (target specific cards/tables containing age groups)
                # We look for text blocks containing numbers and age brackets
                stats_elements = await page.locator(".col-md-6, .card-body, .row").all_inner_texts()

                for text in stats_elements:
                    if "18 - 19" in text or "20 - 29" in text: # Look for the demographic panel
                        structured_data.append({
                            "Province": province,
                            "Municipality": muni,
                            "Raw_Demographics": text.strip()
                        })

        await browser.close()

        # Save clean results
        df_clean = pd.DataFrame(structured_data)
        df_clean.to_csv("iec_clean_demographics.csv", index=False)
        print("Clean extraction complete! Saved to 'iec_clean_demographics.csv'.")

# Run it
await extract_clean_iec_data()

Opening IEC portal...
Processing province: Eastern Cape
  -> Selecting municipality: BUF - Buffalo City
  -> Selecting municipality: EC101 - Dr. Beyers Naude
  -> Selecting municipality: EC102 - Blue Crane Route
  -> Selecting municipality: EC104 - Makana
  -> Selecting municipality: EC105 - Ndlambe
  -> Selecting municipality: EC106 - Sundays River Valley
  -> Selecting municipality: EC108 - Kouga
  -> Selecting municipality: EC109 - Kou-Kamma
  -> Selecting municipality: EC121 - Mbhashe
  -> Selecting municipality: EC122 - Mnquma
  -> Selecting municipality: EC123 - Great Kei
  -> Selecting municipality: EC124 - Amahlathi
  -> Selecting municipality: EC126 - Ngqushwa
  -> Selecting municipality: EC129 - Raymond Mhlaba
  -> Selecting municipality: EC131 - Inxuba Yethemba
  -> Selecting municipality: EC135 - Intsika Yethu
  -> Selecting municipality: EC136 - Emalahleni
  -> Selecting municipality: EC137 - Dr. A.B. Xuma
  -> Selecting municipality: EC138 - Sakhisizwe
  -> Selecting muni


#REGEX-BASED PARSING & DEMOGRAPHIC STRUCTURING PIPELINE
 Objective: Transform unstructured raw text payloads (scraped from the IEC portal) into a clean, tabular, analysis-ready dataset.




In [ ]:
import pandas as pd
import re

# 1. Load your raw demographic text dataset
df_raw = pd.read_csv('iec_clean_demographics.csv')

parsed_data = []
age_groups = ['18 - 19', '20 - 29', '30 - 39', '40 - 49', '50 - 59', '60 - 69', '70 - 79', '80+']

for idx, row in df_raw.iterrows():
    prov = row['Province']
    muni = row['Municipality']
    text = row['Raw_Demographics']

    # Initialize dictionary for the row
    row_dict = {
        'Province': prov,
        'Municipality': muni
    }

    # Extract total registered voters
    total_match = re.search(r'Total\n([0-9,]+)', text)
    row_dict['Total_Registered_Voters'] = int(total_match.group(1).replace(',', '')) if total_match else None

    # Extract overall male and female counts
    overall_male = re.search(r'Overall\nMale\n[^\n]+\n([0-9,]+)', text)
    overall_female = re.search(r'Female\n[^\n]+\n([0-9,]+)', text)

    row_dict['Total_Male'] = int(overall_male.group(1).replace(',', '')) if overall_male else None

    # Extract age-group specific counts (Male & Female)
    for ag in age_groups:
        escaped_ag = re.escape(ag)
        pattern = rf'{escaped_ag}\nMale\n[^\n]+\n([0-9,]+)\nFemale\n[^\n]+\n([0-9,]+)'
        match = re.search(pattern, text)

        if match:
            male_val = int(match.group(1).replace(',', ''))
            female_val = int(match.group(2).replace(',', ''))
            row_dict[f'Male_{ag}'] = male_val
            row_dict[f'Female_{ag}'] = female_val
            row_dict[f'Total_{ag}'] = male_val + female_val
        else:
            row_dict[f'Male_{ag}'] = None
            row_dict[f'Female_{ag}'] = None
            row_dict[f'Total_{ag}'] = None

    parsed_data.append(row_dict)

# Convert to a clean structured DataFrame
df_cleaned = pd.DataFrame(parsed_data)

# Export the cleaned structured dataset to a new CSV file
df_cleaned.to_csv('iec_fully_cleaned_demographics.csv', index=False)

print("Cleaning and parsing complete!")
print(f"Total municipalities processed: {len(df_cleaned)}")
print(df_cleaned[['Province', 'Municipality', 'Total_Registered_Voters', 'Total_18 - 19', 'Total_20 - 29']].head())

Cleaning and parsing complete!
Total municipalities processed: 213
       Province              Municipality  Total_Registered_Voters  \
0  Eastern Cape        BUF - Buffalo City                   458606   
1  Eastern Cape  EC101 - Dr. Beyers Naude                    43220   
2  Eastern Cape  EC102 - Blue Crane Route                    19318   
3  Eastern Cape            EC104 - Makana                    47075   
4  Eastern Cape           EC105 - Ndlambe                    37766   

   Total_18 - 19  Total_20 - 29  
0          10834          69758  
1           1141           7282  
2            666           2922  
3           1290           8863  
4            996           5115  


#WARD-LEVEL GRANULARITY PARSING & ELECTORAL DATA EXTRACTION
Objective: Disaggregate municipal-level scraped payloads down to the granular ward level.



In [ ]:
import pandas as pd
import re

# Load the raw scraped dataset we already saved
df_raw = pd.read_csv('iec_clean_demographics.csv')

ward_rows = []

for idx, row in df_raw.iterrows():
    prov = row['Province']
    muni = row['Municipality']
    text = row['Raw_Demographics']

    # Check if the ward-level table exists in the text block
    if "Registered voters count by province by municipality by ward" in text:
        parts = text.split("Registered voters count by province by municipality by ward")
        ward_section = parts[1]

        # Regex to capture Ward ID, Voting Districts count, and Registered Voters count
        matches = re.findall(r'([0-9]{8})\t([0-9]+)\t([0-9,]+)', ward_section)

        for ward_id, vd_count, reg_voters in matches:
            ward_rows.append({
                'province': prov,
                'municipality': muni,
                'ward_id': ward_id,
                'voting_districts': int(vd_count),
                'registered_people': int(reg_voters.replace(',', ''))
            })

# Convert to DataFrame
df_wards = pd.DataFrame(ward_rows)

# Export to a clean CSV file
df_wards.to_csv('iec_registered_people_per_ward.csv', index=False)

print(f"Successfully extracted {len(df_wards)} wards across all municipalities!")
print(df_wards.head())

Successfully extracted 3094 wards across all municipalities!
       province        municipality   ward_id  voting_districts  \
0  Eastern Cape  BUF - Buffalo City  29200001                 5   
1  Eastern Cape  BUF - Buffalo City  29200002                 5   
2  Eastern Cape  BUF - Buffalo City  29200003                 3   
3  Eastern Cape  BUF - Buffalo City  29200004                 4   
4  Eastern Cape  BUF - Buffalo City  29200005                 6   

   registered_people  
0              10526  
1               8040  
2              12474  
3              10247  
4               8864  
